<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🚀 تحسين بناء النموذج (Improved Model Building)</div>
    <div class="eda-text-box">
        في هذه المرحلة يتم العمل على
        <span class="eda-highlight">تحسين بنية النموذج</span>
        الذي تم إنشاؤه سابقًا، وذلك بهدف رفع كفاءة التعلم
        وتحسين الأداء العام للنموذج على بيانات الاختبار.
    </div>
    <div class="eda-text-box">
        يركز هذا التحسين على الاستفادة بشكل أعمق من النموذج
        المدرّب مسبقًا، من خلال
        <span class="eda-highlight">تعديل استراتيجية التدريب</span>
        أو تحسين طريقة التعامل مع الطبقات القابلة للتدريب،
        دون المساس بالمعرفة الأساسية المكتسبة من ImageNet.
    </div>
    <div class="eda-text-box">
        تسمح هذه الخطوة بتحقيق توازن أفضل بين
        <span class="eda-highlight">المرونة والتعميم</span>،
        حيث يصبح النموذج أكثر قدرة على التكيّف مع بيانات
        المشروع مع الحفاظ على الاستقرار وتقليل خطر
        فرط التعلّم (Overfitting).
    </div>

</div>

<div class="eda-note">
    ℹ️ تهدف هذه المرحلة إلى تحسين أداء النموذج النهائي دون زيادة كبيرة في التعقيد أو استهلاك الموارد.
</div>

</div>


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix
import plotly.graph_objects as go
import plotly.express as px

DATA_ROOT = "datasat orginal"   # مسار البيانات الأصلية
IMG_SIZE = (224,224)
BATCH = 32
EPOCHS_FAST = 3
EPOCHS_FINE = 40\
MODEL_PATH = "best_model.keras"
AUTOTUNE = tf.data.AUTOTUNE


<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🧹 تجهيز ومعالجة مجموعات البيانات (Dataset Preparation)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم تنفيذ خطوة
        <span class="eda-highlight">تهيئة البيانات قبل التدريب</span>،
        حيث يتم تجهيز الصور والعناوين الخاصة بها بصيغة مناسبة
        لاستخدامها مباشرة داخل نموذج
        EfficientNet.
    </div>
    <div class="eda-text-box">
        يتم تعريف دالة
        <span class="eda-highlight">preprocess</span>
        لمعالجة الصور، والتي تشمل تحويل نوع البيانات إلى
        <span class="eda-highlight">float32</span>
        وتطبيق دالة المعالجة المسبقة الخاصة بـ EfficientNet،
        مما يضمن توافق البيانات مع النموذج المدرّب مسبقًا.
    </div>
    <div class="eda-text-box">
        تعتمد دالة
        <span class="eda-highlight">prepare_ds</span>
        على تحميل الصور من المجلدات باستخدام
        <span class="eda-highlight">image_dataset_from_directory</span>،
        مع فرض استخدام نظام الألوان
        <span class="eda-highlight">RGB</span>
        بثلاث قنوات، لضمان توحيد شكل البيانات
        وإسقاط أي قنوات غير مرغوب فيها.
    </div>
    <div class="eda-text-box">
        قبل تطبيق أي معالجة إضافية، يتم حفظ
        <span class="eda-highlight">أسماء الفئات (class names)</span>
        مباشرة من مجموعة البيانات الخام، وذلك لاستخدامها لاحقًا
        في التفسير أو عرض النتائج.
    </div>
    <div class="eda-text-box">
        أخيرًا، يتم تطبيق المعالجة المسبقة باستخدام
        <span class="eda-highlight">map</span>
        مع تفعيل
        <span class="eda-highlight">AUTOTUNE</span>
        لتحسين الأداء، بالإضافة إلى استخدام
        <span class="eda-highlight">prefetch</span>
        لتسريع عملية تحميل البيانات أثناء التدريب.
    </div>

</div>

<div class="eda-note">
    ℹ️ هذه الخطوة تضمن أن جميع مجموعات البيانات (التدريب، التحقق، الاختبار) جاهزة ومتوافقة مع النموذج من حيث الشكل والأداء.
</div>

</div>


In [22]:
import os
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

# def fix_channels(image):                               # [محذوف] لم نعد بحاجة لتصحيح القنوات يدويًا
#     ...

def preprocess(image, label):
    image = tf.cast(image, tf.float32)                   # كما هو/نُبقي عليه
    # image = fix_channels(image)                        # [محذوف] لأننا سنفرض RGB من المصدر
    image = tf.keras.applications.efficientnet.preprocess_input(image)  # كما هو
    return image, label

def prepare_ds(split):
    raw = tf.keras.preprocessing.image_dataset_from_directory(
        f"{DATA_ROOT}/{split}",
        image_size=IMG_SIZE,
        batch_size=BATCH,
        label_mode="categorical",
        shuffle=True,
        color_mode="rgb"                                  # [جديد] فرض RGB بثلاث قنوات وإسقاط أي قنوات زائدة
    )
    class_names = raw.class_names                         # [جديد] حفظ أسماء الفئات قبل أي map
    ds = raw.map(preprocess, num_parallel_calls=AUTOTUNE) # [استبدال] نعمل map على raw
    ds = ds.prefetch(AUTOTUNE)                            # كما هو
    return ds, class_names                                # [جديد] نعيد أيضًا class_names

# train_ds = prepare_ds("Train")                         # [محذوف] أصبح prepare_ds يعيد قيمتين
# val_ds   = prepare_ds("Validate")                      # [محذوف]
# test_ds  = prepare_ds("Test")                          # [محذوف]

train_ds, class_names = prepare_ds("Train")              # [بديل جديد]
val_ds,   _           = prepare_ds("Validate")           # [بديل جديد]
test_ds,  _           = prepare_ds("Test")               # [بديل جديد]

# class_names = train_ds.class_names                     # [محذوف] train_ds الآن PrefetchDataset ولا يملك class_names
# len(class_names), class_names                          # كما تريد لاحقًا: استخدم المتغير class_names الموجود

Found 21234 files belonging to 11 classes.
Found 2650 files belonging to 11 classes.
Found 2650 files belonging to 11 classes.


<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🎨 تعزيز البيانات (Data Augmentation)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم تطبيق تقنية
        <span class="eda-highlight">تعزيز البيانات</span>،
        والتي تهدف إلى زيادة تنوّع بيانات التدريب بشكل اصطناعي،
        مما يساعد النموذج على التعميم بشكل أفضل وتقليل
        مشكلة فرط التعلّم (Overfitting).
    </div>
    <div class="eda-text-box">
        يتم إنشاء تسلسل من التحويلات العشوائية باستخدام
        <span class="eda-highlight">Keras Sequential API</span>،
        وتشمل عمليات مثل قلب الصور أفقيًا، التدوير،
        التكبير، وتغيير التباين، مع الحفاظ على
        البنية العامة للصورة.
    </div>
    <div class="eda-text-box">
        تُستخدم دالة
        <span class="eda-highlight">augment</span>
        لتطبيق عمليات التعزيز على الصور فقط أثناء
        مرحلة التدريب، من خلال تفعيل الوضع
        <span class="eda-highlight">training=True</span>،
        دون التأثير على التسميات (Labels).
    </div>
    <div class="eda-text-box">
        أخيرًا، يتم تطبيق التعزيز على مجموعة بيانات
        التدريب باستخدام
        <span class="eda-highlight">map</span>
        مع تفعيل
        <span class="eda-highlight">AUTOTUNE</span>
        لتحسين كفاءة التنفيذ وتسريع عملية التحميل.
    </div>

</div>

<div class="eda-note">
    ℹ️ يساهم تعزيز البيانات في جعل النموذج أكثر قدرة على التعامل مع التغيّرات البصرية في البيانات الحقيقية.
</div>

</div>


In [23]:
data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

def augment(image, label):
    return data_aug(image, training=True), label

train_ds_aug = train_ds.map(augment, num_parallel_calls=AUTOTUNE)


<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🧠 إنشاء وتجميع النموذج (Model Construction & Compilation)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم إنشاء النموذج النهائي باستخدام
        <span class="eda-highlight">تقنية التعلم بالنقل (Transfer Learning)</span>،
        حيث يتم الاعتماد على نموذج
        <span class="eda-highlight">EfficientNetB3</span>
        المدرّب مسبقًا على بيانات ImageNet كمستخرج للخصائص.
    </div>
    <div class="eda-text-box">
        يتم تحميل النموذج الأساسي بدون الطبقات النهائية
        (<span class="eda-highlight">include_top=False</span>)
        وتجميد جميع أوزانه، وذلك للحفاظ على المعرفة
        المكتسبة مسبقًا ومنع تحديثها أثناء التدريب.
    </div>
    <div class="eda-text-box">
        بعد ذلك يتم تعريف مدخلات النموذج وتمريرها عبر
        النموذج الأساسي في وضع
        <span class="eda-highlight">training=False</span>،
        ثم تطبيق طبقة
        <span class="eda-highlight">Global Average Pooling</span>
        لاستخلاص الميزات عالية المستوى.
    </div>
    <div class="eda-text-box">
        تتم إضافة طبقة
        <span class="eda-highlight">Dropout</span>
        بنسبة 0.3 لتقليل فرط التعلّم، يليها طبقة
        <span class="eda-highlight">Dense</span>
        نهائية بعدد فئات يساوي عدد أسماء الفئات،
        مع استخدام دالة التفعيل
        <span class="eda-highlight">softmax</span>
        لإخراج احتمالات التصنيف.
    </div>
    <div class="eda-text-box">
        أخيرًا يتم تجميع النموذج باستخدام
        <span class="eda-highlight">Adam Optimizer</span>
        ودالة خسارة
        <span class="eda-highlight">categorical_crossentropy</span>،
        مع اعتماد الدقة كمقياس للأداء، ثم عرض ملخص
        النموذج للتحقق من بنيته.
    </div>

</div>

<div class="eda-note">
    ℹ️ تمثل هذه الخلية القلب الأساسي للمشروع، حيث يتم فيها إنشاء النموذج الذي سيخضع لعملية التدريب والتقييم.
</div>

</div>


In [24]:
base = tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 11)             │        16,907 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,800,442 (41.20 MB)

 Trainable params: 16,907 (66.04 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">⚡ التدريب السريع للنموذج (Initial Training)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم تنفيذ مرحلة
        <span class="eda-highlight">التدريب الأولي</span>
        للنموذج، حيث يتم تدريب الطبقات المضافة حديثًا فقط
        مع إبقاء النموذج الأساسي مجمّدًا.
    </div>
    <div class="eda-text-box">
        يعتمد التدريب على بيانات التدريب المعززة
        <span class="eda-highlight">train_ds_aug</span>
        مع استخدام بيانات التحقق لمتابعة أداء النموذج
        خلال عدد محدود من العصور التدريبية.
    </div>

</div>

<div class="eda-note">
    ℹ️ هذه المرحلة تهيئ النموذج للانتقال إلى مراحل تدريب أكثر تقدمًا.
</div>

</div>


In [25]:
history_fast = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=EPOCHS_FAST
)

Epoch 1/3
664/664 ━━━━━━━━━━━━━━━━━━━━ 870s 1s/step - accuracy: 0.5959 - loss: 1.1626 - val_accuracy: 0.6830 - val_loss: 0.8978
Epoch 2/3
664/664 ━━━━━━━━━━━━━━━━━━━━ 764s 1s/step - accuracy: 0.7066 - loss: 0.8135 - val_accuracy: 0.7358 - val_loss: 0.7425
Epoch 3/3
664/664 ━━━━━━━━━━━━━━━━━━━━ 768s 1s/step - accuracy: 0.7346 - loss: 0.7349 - val_accuracy: 0.7675 - val_loss: 0.6838


In [26]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history_fast.history['accuracy'], name="Train Acc"))
fig.add_trace(go.Scatter(y=history_fast.history['val_accuracy'], name="Val Acc"))
fig.update_layout(title="Quick Training Accuracy", template="plotly_white")
fig.show()


<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">🎯 الضبط الدقيق للنموذج (Fine-Tuning)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم تنفيذ مرحلة
        <span class="eda-highlight">الضبط الدقيق (Fine-Tuning)</span>،
        حيث يتم فك تجميد جزء من طبقات النموذج الأساسي
        لتحديث أوزانها بشكل محدود وتحسين الأداء النهائي.
    </div>
    <div class="eda-text-box">
        يتم إعادة تجميع النموذج باستخدام
        <span class="eda-highlight">معدل تعلم منخفض</span>
        لضمان إجراء تعديلات دقيقة على الأوزان دون التأثير
        سلبًا على المعرفة المكتسبة مسبقًا.
    </div>
    <div class="eda-text-box">
        يتم تدريب النموذج باستخدام بيانات التدريب المعززة
        مع مراقبة أداء التحقق، مع تفعيل
        <span class="eda-highlight">حفظ أفضل نموذج</span>
        والإيقاف المبكر لتجنب فرط التعلّم.
    </div>

</div>

<div class="eda-note">
    ℹ️ تساهم هذه المرحلة في تحسين دقة النموذج مع الحفاظ على استقراره.
</div>

</div>


In [27]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_accuracy"),
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)


Epoch 1/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 961s 1s/step - accuracy: 0.7290 - loss: 0.7505 - val_accuracy: 0.7951 - val_loss: 0.5676
Epoch 2/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 19880s 30s/step - accuracy: 0.7721 - loss: 0.6070 - val_accuracy: 0.8075 - val_loss: 0.5123
Epoch 3/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 988s 1s/step - accuracy: 0.7898 - loss: 0.5584 - val_accuracy: 0.8211 - val_loss: 0.4807
Epoch 4/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 994s 1s/step - accuracy: 0.8033 - loss: 0.5183 - val_accuracy: 0.8279 - val_loss: 0.4542
Epoch 5/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 979s 1s/step - accuracy: 0.8064 - loss: 0.4994 - val_accuracy: 0.8317 - val_loss: 0.4375
Epoch 6/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 986s 1s/step - accuracy: 0.8198 - loss: 0.4675 - val_accuracy: 0.8449 - val_loss: 0.4116
Epoch 7/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 984s 1s/step - accuracy: 0.8283 - loss: 0.4529 - val_accuracy: 0.8513 - val_loss: 0.3952
Epoch 8/40
664/664 ━━━━━━━━━━━━━━━━━━━━ 987s 1s/step - accuracy: 0.8338 - loss: 0.4329 - val_a

In [28]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=history_fine.history['accuracy'], name="Train Fine"))
fig.add_trace(go.Scatter(y=history_fine.history['val_accuracy'], name="Val Fine"))
fig.update_layout(title="Fine-Tuning Accuracy", template="plotly_white")
fig.show()


<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">📊 تقييم النموذج باستخدام مصفوفة الالتباس</div>
    <div class="eda-text-box">
        في هذه الخلية يتم تحميل
        <span class="eda-highlight">أفضل نسخة من النموذج</span>
        التي تم حفظها أثناء التدريب، ثم استخدامها لإجراء
        التنبؤات على مجموعة بيانات الاختبار.
    </div>
    <div class="eda-text-box">
        يتم استخراج القيم الحقيقية والمتوقعة للفئات،
        ومن ثم حساب
        <span class="eda-highlight">مصفوفة الالتباس</span>
        ونسختها المطبّعة، مما يساعد على تحليل أداء
        النموذج لكل فئة بشكل مستقل.
    </div>
    <div class="eda-text-box">
        أخيرًا يتم عرض مصفوفة الالتباس المطبّعة باستخدام
        <span class="eda-highlight">Plotly</span>،
        لتوفير تمثيل بصري واضح يسهّل تفسير
        الأخطاء ونقاط القوة في النموذج.
    </div>

</div>

<div class="eda-note">
    ℹ️ تساعد مصفوفة الالتباس في فهم سلوك النموذج وتحديد الفئات التي تحتاج إلى تحسين.
</div>

</div>


In [29]:
model = tf.keras.models.load_model(MODEL_PATH)

y_true = []
y_pred = []

for x,y in test_ds:
    preds = model.predict(x)
    y_true.extend(tf.argmax(y, axis=1).numpy())
    y_pred.extend(tf.argmax(preds, axis=1).numpy())

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig = px.imshow(
    cm_norm,
    x=class_names,
    y=class_names,
    text_auto=".2f",
    color_continuous_scale="Blues",
    title="Normalized Confusion Matrix"
)
fig.update_layout(template="plotly_white", height=700)
fig.show()


1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 

<style>
/* ===== PROJECT GLOBAL STYLE ===== */
.eda-intro {
    direction: rtl;
    text-align: right;
    font-family: 'Arial', sans-serif;
    margin: 30px auto;
    max-width: 1400px;
}

.eda-container {
    background: #E3F2FD;
    border-radius: 10px;
    padding: 22px;
    margin-bottom: 18px;
    border: 1.5px solid #90CAF9;
}

.eda-title {
    color: #1565C0;
    border-right: 4px solid #42A5F5;
    padding-right: 14px;
    margin-bottom: 18px;
    font-size: 1.55em;
    font-weight: bold;
}

.eda-text-box {
    background: #F1F8FF;
    padding: 14px 16px;
    margin: 12px 0;
    border-radius: 8px;
    border-right: 3px solid #64B5F6;
    color: #0D47A1;
    line-height: 1.9;
    font-size: 0.98em;
}

.eda-highlight {
    color: #1565C0;
    font-weight: bold;
}

.eda-note {
    color: #0D47A1;
    font-weight: bold;
    margin-top: 20px;
    padding: 12px;
    background-color: #E3F2FD;
    border: 1.5px dashed #64B5F6;
    border-radius: 8px;
    text-align: center;
    font-size: 0.95em;
}
</style>

<div class="eda-intro">

<div class="eda-container">
    <div class="eda-title">💾 حفظ النموذج وأسماء الفئات (Model Saving)</div>
    <div class="eda-text-box">
        في هذه الخلية يتم حفظ
        <span class="eda-highlight">النموذج النهائي</span>
        بصيغة Keras الحديثة، وهي الصيغة الموصى بها
        لاستخدامها لاحقًا في النشر أو إعادة التحميل.
    </div>
    <div class="eda-text-box">
        يتم كذلك استخراج
        <span class="eda-highlight">أسماء الفئات</span>
        من مجلدات بيانات التدريب وحفظها في ملف
        <span class="eda-highlight">JSON</span>،
        لضمان استخدامها لاحقًا أثناء التنبؤ أو عرض النتائج.
    </div>
    <div class="eda-text-box">
        يتضمن الكود آلية مرنة للتعامل مع الحالات التي
        لا يكون فيها النموذج أو البيانات محمّلة مسبقًا،
        مما يزيد من
        <span class="eda-highlight">قابلية إعادة الاستخدام</span>
        والاعتمادية.
    </div>

</div>

<div class="eda-note">
    ℹ️ حفظ النموذج مع أسماء الفئات يسهّل عملية النشر وإعادة الاستخدام دون الحاجة لإعادة التدريب.
</div>

</div>


In [30]:
# خلية A: حفظ النموذج بصيغة Keras الحديثة وحفظ أسماء الفئات
import json
from pathlib import Path
import tensorflow as tf

# مسارات
MODEL_PATH = Path("best_modelt.keras")   # ملف النموذج النهائي
CLASSN_PATH = Path("class_names.json")  # حفظ أسماء الفئات

# إذا كان النموذج موجودًا في الذاكرة كـ `model`، احفظه. وإلا حمّله من مسار آخر.
try:
    model  # إذا موجود في الذاكرة
except NameError:
    # حاول تحميل النموذج المحفوظ مسبقًا (قد يكون 'models/efficientnetb3_wbc.h5' أو ما شابه)
    # غيّر المسار إن لزم
    model = tf.keras.models.load_model("models/efficientnetb3_wbc.h5")

# احفظ بصيغة Keras الحديثة (مستحسن)
model.save(str(MODEL_PATH), save_format="keras")
print("✅ Model saved to", MODEL_PATH)

# احصل على أسماء الفئات من dataset (إن لم تكن محفوظة مسبقاً)
# تأكد أن DATA_ROOT/train يحتوي مجلدات الفئات
DATA_ROOT = Path("dataset")
train_folder = DATA_ROOT / "train"
if train_folder.exists():
    class_names = sorted([d.name for d in train_folder.iterdir() if d.is_dir()])
else:
    # إذا لم نجد dataset، جرّب أن تأخذ أسماء الفئات من model (إن كانت محفوظة في مكان آخر)
    # أو من المتغير train_ds إن كنت تستخدم tf.data (مثال: train_ds.class_names)
    try:
        class_names = model.layers[-1].get_config().get("units")  # fallback dummy
    except:
        class_names = []

# إذا وجدت أسماء جيدة احفظها
if class_names:
    with open(CLASSN_PATH, "w", encoding="utf-8") as f:
        json.dump(class_names, f, ensure_ascii=False, indent=2)
    print("✅ class names saved to", CLASSN_PATH)
else:
    print("⚠️ class names not found automatically — رجاءً أنشئ class_names.json يدوياً إذا لزم.")


✅ Model saved to best_modelt.keras
✅ class names saved to class_names.json
